# TLC Trip Record Data — Ingesta Nivel 2 y Nivel 3

Pipeline de ingesta para los archivos mensuales de TLC.

**Nivel 2**
- Catálogo de archivos.
- Descarga idempotente.
- Validación de metadata.
- Manifest y auditoría.

**Nivel 3**
- Zona Raw con archivos Parquet originales.
- Zona Bronze con archivos Parquet organizados y metadata técnica.
- Validación con PySpark obligatoria.

El notebook no lee la página HTML de NYC.gov para evitar bloqueos `403`. Construye las rutas públicas de descarga de TLC usando el patrón oficial de archivos Parquet.

In [ ]:
# ============================================================
# 0. PARÁMETROS DE EJECUCIÓN
# ============================================================

from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import platform

YEARS = [2023, 2024, 2025]
TRIP_TYPES = ["all"]          # ["all"] = yellow, green, fhv, fhvhv
MAX_FILES_FOR_TEST = None       # None = ejecución completa
OVERWRITE_RAW = False
STRICT_COMPLETENESS = True

LOCAL_TZ = "America/Lima"
PIPELINE_NAME = "tlc_ingestion_nivel_2_3"
PIPELINE_VERSION = "1.0.0"

BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
SOURCE_PAGE = "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page"

TRIP_TYPE_PREFIX = {
    "yellow": "yellow_tripdata",
    "green": "green_tripdata",
    "fhv": "fhv_tripdata",
    "fhvhv": "fhvhv_tripdata",
}

CLOSED_YEARS = [year for year in YEARS if year in [2023, 2024, 2025]]
DYNAMIC_YEARS = [year for year in YEARS if year >= 2026]

if platform.system().lower() == "windows":
    WORKSPACE_DIR = Path(r"../../data")
else:
    WORKSPACE_DIR = Path.cwd() / "tlc_ingestion_workspace"

RAW_DIR = WORKSPACE_DIR / "raw"
BRONZE_DIR = WORKSPACE_DIR / "bronze"
BRONZE_FILES_DIR = BRONZE_DIR / "files"
BRONZE_METADATA_DIR = BRONZE_DIR / "_metadata"
LOG_DIR = WORKSPACE_DIR / "logs"
SPARK_STAGING_DIR = WORKSPACE_DIR / "_spark_staging"
# ============================================================
# Taxi Zone Lookup Table
# ============================================================

INCLUDE_LOOKUP = True

LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

LOOKUP_EXPECTED_COLUMNS = [
    "LocationID",
    "Borough",
    "Zone",
    "service_zone",
]

LOOKUP_RAW_PATH = RAW_DIR / "lookup" / "taxi_zone_lookup.csv"
LOOKUP_BRONZE_PATH = BRONZE_FILES_DIR / "lookup" / "taxi_zone_lookup.csv"

LOOKUP_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
LOOKUP_BRONZE_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_TS = datetime.now(ZoneInfo(LOCAL_TZ)).isoformat()
RUN_ID = datetime.now(ZoneInfo(LOCAL_TZ)).strftime("tlc_ingestion_%Y%m%d_%H%M%S")
RUN_DATE = datetime.now(ZoneInfo(LOCAL_TZ)).strftime("%Y-%m-%d")
RUN_TIME = datetime.now(ZoneInfo(LOCAL_TZ)).strftime("%H:%M:%S")

MANIFEST_JSONL = LOG_DIR / "tlc_ingestion_manifest.jsonl"

for directory in [RAW_DIR, BRONZE_FILES_DIR, BRONZE_METADATA_DIR, LOG_DIR, SPARK_STAGING_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

In [12]:
# ============================================================
# 1. UTILIDADES DE PRESENTACIÓN Y VALIDACIÓN
# ============================================================

import os
import json
import shutil
import hashlib
import subprocess
import urllib.request
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

try:
    from tqdm import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

console = Console()

def panel(title, message, style="blue"):
    console.print(Panel(str(message), title=title, style=style))

def format_table_value(value):
    if value is None:
        return ""

    if isinstance(value, (list, tuple, set)):
        return str(list(value))

    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)

    try:
        missing = pd.isna(value)
        if isinstance(missing, bool) and missing:
            return ""
    except Exception:
        pass

    return str(value)

def dataframe_table(df, title, max_rows=40):
    table = Table(title=title)

    for column in df.columns:
        table.add_column(str(column), overflow="fold")

    show_df = df.head(max_rows).copy()

    for _, row in show_df.iterrows():
        table.add_row(*[format_table_value(value) for value in row])

    console.print(table)

    if len(df) > max_rows:
        console.print(f"Mostrando {max_rows} de {len(df)} registros.")

def normalize_trip_types(trip_types):
    if "all" in trip_types:
        return list(TRIP_TYPE_PREFIX.keys())

    invalid = [item for item in trip_types if item not in TRIP_TYPE_PREFIX]
    if invalid:
        raise ValueError(f"Tipos inválidos: {invalid}")

    return trip_types

SELECTED_TRIP_TYPES = normalize_trip_types(TRIP_TYPES)

config_df = pd.DataFrame([
    ["YEARS", YEARS],
    ["TRIP_TYPES", TRIP_TYPES],
    ["SELECTED_TRIP_TYPES", SELECTED_TRIP_TYPES],
    ["MAX_FILES_FOR_TEST", MAX_FILES_FOR_TEST],
    ["OVERWRITE_RAW", OVERWRITE_RAW],
    ["STRICT_COMPLETENESS", STRICT_COMPLETENESS],
    ["WORKSPACE_DIR", WORKSPACE_DIR],
    ["RAW_DIR", RAW_DIR],
    ["BRONZE_DIR", BRONZE_DIR],
    ["MANIFEST_JSONL", MANIFEST_JSONL],
], columns=["parámetro", "valor"])

dataframe_table(config_df, "Parámetros activos", max_rows=20)

                          Parámetros activos                          
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ parámetro           ┃ valor                                        ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ YEARS               │ [2023]                                       │
│ TRIP_TYPES          │ ['all']                                      │
│ SELECTED_TRIP_TYPES │ ['yellow', 'green', 'fhv', 'fhvhv']          │
│ MAX_FILES_FOR_TEST  │ 4                                            │
│ OVERWRITE_RAW       │ False                                        │
│ STRICT_COMPLETENESS │ True                                         │
│ WORKSPACE_DIR       │ ..\..\data                                   │
│ RAW_DIR             │ ..\..\data\raw                               │
│ BRONZE_DIR          │ ..\..\data\bronze                            │
│ MANIFEST_JSONL      │ ..\..\data\logs\tlc_ingestion_manifest.jsonl │
└─────────────────────┴──────────────────────────────────────────────┘

In [13]:
# ============================================================
# 2. CONFIGURACIÓN DE JAVA Y PYSPARK
# ============================================================

import os
from pathlib import Path

def first_existing_path(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return str(path)
    return None

if platform.system().lower() == "windows":
    detected_java = first_existing_path([
        r"C:\Program Files\Java\jdk-17",
        r"C:\Program Files\Java\jdk-21",
        r"C:\Program Files\Eclipse Adoptium\jdk-17",
        r"C:\Program Files\Eclipse Adoptium\jdk-21",
    ])

    if detected_java:
        os.environ["JAVA_HOME"] = detected_java
        os.environ["PATH"] = detected_java + r"\bin;" + os.environ["PATH"]

    if Path(r"C:\hadoop\bin").exists():
        os.environ["HADOOP_HOME"] = r"C:\hadoop"
        os.environ["hadoop.home.dir"] = r"C:\hadoop"
        os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

panel("JAVA_HOME", os.environ.get("JAVA_HOME", "No configurado"), "blue")
panel("HADOOP_HOME", os.environ.get("HADOOP_HOME", "No configurado. No se usará Spark write local."), "blue")

╭─────────────────────────────────────────────────── JAVA_HOME ───────────────────────────────────────────────────╮
│ C:\Program Files\Java\jdk-17                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── HADOOP_HOME ──────────────────────────────────────────────────╮
│ No configurado. No se usará Spark write local.                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
# ============================================================
# 3. SPARK SESSION
# ============================================================

import time
from pyspark.sql import SparkSession

spark_start = time.perf_counter()

spark = (
    SparkSession.builder
    .appName("TLC_Ingestion_Nivel_2_3")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.eventLog.enabled", "false")
    .config("spark.sql.shuffle.partitions", "1")
    .config("spark.default.parallelism", "1")
    .config("spark.sql.parquet.mergeSchema", "false")
    .config("spark.sql.files.ignoreCorruptFiles", "false")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.range(1).collect()

elapsed = round(time.perf_counter() - spark_start, 2)
panel("Spark activo", f"Versión: {spark.version} | Inicio: {elapsed} segundos", "green")

╭───────────────────────────────────────────────── Spark activo ──────────────────────────────────────────────────╮
│ Versión: 4.1.2 | Inicio: 0.07 segundos                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
# ============================================================
# 4. CATÁLOGO DE ARCHIVOS TLC
# ============================================================

from datetime import datetime

def build_file_record(year, month, trip_type):
    prefix = TRIP_TYPE_PREFIX[trip_type]
    file_name = f"{prefix}_{year}-{month:02d}.parquet"
    url = f"{BASE_URL}/{file_name}"

    local_path = (
        RAW_DIR
        / f"tipo_dataset={trip_type}"
        / f"anio={year}"
        / f"mes={month:02d}"
        / file_name
    )

    return {
        "year": int(year),
        "month": int(month),
        "trip_type": trip_type,
        "file_name": file_name,
        "url": url,
        "local_path": str(local_path),
    }

def url_exists_with_curl(url):
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": SOURCE_PAGE,
    }

    if shutil.which("curl") is not None:
        cmd = [
            "curl",
            "-L",
            "--fail",
            "--silent",
            "--show-error",
            "--head",
            "--connect-timeout", "20",
            "--max-time", "40",
            "-A", "Mozilla/5.0",
            "-e", SOURCE_PAGE,
            url,
        ]

        completed = subprocess.run(cmd, capture_output=True, text=True)
        return completed.returncode == 0

    try:
        request = urllib.request.Request(
            url,
            method="HEAD",
            headers=headers,
        )

        with urllib.request.urlopen(request, timeout=40) as response:
            return 200 <= response.status < 400

    except Exception:
        return False

def build_catalog():
    records = []
    current_year = datetime.now().year
    current_month = datetime.now().month

    for year in YEARS:
        for month in range(1, 13):
            if year > current_year:
                continue

            if year == current_year and month > current_month:
                continue

            for trip_type in SELECTED_TRIP_TYPES:
                record = build_file_record(year, month, trip_type)

                if year in DYNAMIC_YEARS:
                    exists = url_exists_with_curl(record["url"])
                    if exists is False:
                        continue

                records.append(record)

    return pd.DataFrame(records)

catalog_df = build_catalog()

summary_df = (
    catalog_df
    .groupby(["year", "trip_type"], as_index=False)
    .agg(files=("file_name", "count"))
    .sort_values(["year", "trip_type"])
)

dataframe_table(summary_df, "Catálogo detectado", max_rows=80)
dataframe_table(catalog_df, "Archivos catalogados", max_rows=40)

     Catálogo detectado     
┏━━━━━━┳━━━━━━━━━━━┳━━━━━━━┓
┃ year ┃ trip_type ┃ files ┃
┡━━━━━━╇━━━━━━━━━━━╇━━━━━━━┩
│ 2023 │ fhv       │ 12    │
│ 2023 │ fhvhv     │ 12    │
│ 2023 │ green     │ 12    │
│ 2023 │ yellow    │ 12    │
└──────┴───────────┴───────┘

                                               Archivos catalogados                                                
┏━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ year ┃ month ┃ trip_type ┃ file_name                  ┃ url                        ┃ local_path                 ┃
┡━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 2023 │ 1     │ yellow    │ yellow_tripdata_2023-01.pa │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ rquet                      │ udfront.net/trip-data/yell │ t=yellow\anio=2023\mes=01\ │
│      │       │           │                            │ ow_tripdata_2023-01.parque │ yellow_tripdata_2023-01.pa │
│      │       │           │                            │ t                          │ rquet                      │
│ 2023 │ 1     │ green     │ green_tripdata_2023-01.par │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ quet                       │ udfront.net/trip-data/gree │ t=green\anio=2023\mes=01\g │
│      │       │           │                            │ n_tripdata_2023-01.parquet │ reen_tripdata_2023-01.parq │
│      │       │           │                            │                            │ uet                        │
│ 2023 │ 1     │ fhv       │ fhv_tripdata_2023-01.parqu │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ et                         │ udfront.net/trip-data/fhv_ │ t=fhv\anio=2023\mes=01\fhv │
│      │       │           │                            │ tripdata_2023-01.parquet   │ _tripdata_2023-01.parquet  │
│ 2023 │ 1     │ fhvhv     │ fhvhv_tripdata_2023-01.par │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ quet                       │ udfront.net/trip-data/fhvh │ t=fhvhv\anio=2023\mes=01\f │
│      │       │           │                            │ v_tripdata_2023-01.parquet │ hvhv_tripdata_2023-01.parq │
│      │       │           │                            │                            │ uet                        │
│ 2023 │ 2     │ yellow    │ yellow_tripdata_2023-02.pa │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ rquet                      │ udfront.net/trip-data/yell │ t=yellow\anio=2023\mes=02\ │
│      │       │           │                            │ ow_tripdata_2023-02.parque │ yellow_tripdata_2023-02.pa │
│      │       │           │                            │ t                          │ rquet                      │
│ 2023 │ 2     │ green     │ green_tripdata_2023-02.par │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ quet                       │ udfront.net/trip-data/gree │ t=green\anio=2023\mes=02\g │
│      │       │           │                            │ n_tripdata_2023-02.parquet │ reen_tripdata_2023-02.parq │
│      │       │           │                            │                            │ uet                        │
│ 2023 │ 2     │ fhv       │ fhv_tripdata_2023-02.parqu │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ et                         │ udfront.net/trip-data/fhv_ │ t=fhv\anio=2023\mes=02\fhv │
│      │       │           │                            │ tripdata_2023-02.parquet   │ _tripdata_2023-02.parquet  │
│ 2023 │ 2     │ fhvhv     │ fhvhv_tripdata_2023-02.par │ https://d37ci6vzurychx.clo │ ..\..\data\raw\tipo_datase │
│      │       │           │ quet                       │ udfront.net/trip-data/fhvh │ t=fhvhv\anio=2023\mes=02\f │
│      │       │           │                            │ v_tripdata_2023-02.parquet │ hvhv_tripdata_2023-02.parq │
│      │       │           │                            │                            │ uet                        │
│ 2023 │ 3     │ yellow    │ yellow_tripdata_2023-03.pa 

Mostrando 40 de 48 registros.

In [16]:
# ============================================================
# 5. VALIDACIÓN DE COBERTURA
# ============================================================

expected_records = []

for year in CLOSED_YEARS:
    for month in range(1, 13):
        for trip_type in SELECTED_TRIP_TYPES:
            expected_records.append({
                "year": int(year),
                "month": int(month),
                "trip_type": trip_type,
            })

expected_df = pd.DataFrame(expected_records)

if expected_df.empty:
    panel("Validación de cobertura", "No hay años cerrados para validar.", "yellow")
else:
    current_df = catalog_df[["year", "month", "trip_type"]].drop_duplicates()

    missing_df = (
        expected_df
        .merge(current_df, on=["year", "month", "trip_type"], how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns=["_merge"])
    )

    if missing_df.empty:
        panel("Validación de cobertura", "Cobertura completa para años cerrados.", "green")
    else:
        dataframe_table(missing_df, "Faltantes en años cerrados", max_rows=80)

        if STRICT_COMPLETENESS:
            raise RuntimeError(f"Se detectaron {len(missing_df)} archivo(s) faltantes en años cerrados.")
        else:
            panel("Validación de cobertura", "Hay faltantes, pero STRICT_COMPLETENESS=False.", "yellow")

╭──────────────────────────────────────────── Validación de cobertura ────────────────────────────────────────────╮
│ Cobertura completa para años cerrados.                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:
# ============================================================
# 6. DESCARGA RAW
# ============================================================

def has_curl():
    return shutil.which("curl") is not None

def download_with_curl(url, target_path):
    cmd = [
        "curl",
        "-L",
        "--fail",
        "--silent",
        "--show-error",
        "--retry", "3",
        "--retry-delay", "2",
        "--connect-timeout", "30",
        "--max-time", "0",
        "-A", "Mozilla/5.0",
        "-e", SOURCE_PAGE,
        "-o", str(target_path),
        url,
    ]

    completed = subprocess.run(cmd, capture_output=True, text=True)

    if completed.returncode != 0:
        err = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(err)

def download_with_urllib(url, target_path):
    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Mozilla/5.0",
            "Referer": SOURCE_PAGE,
        },
    )

    with urllib.request.urlopen(request, timeout=120) as response:
        with open(target_path, "wb") as file:
            shutil.copyfileobj(response, file)

def sha256_file(file_path, chunk_size=1024 * 1024 * 8):
    h = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

def append_manifest(record):
    with open(MANIFEST_JSONL, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

download_df = catalog_df.copy()

if MAX_FILES_FOR_TEST is not None:
    download_df = download_df.head(MAX_FILES_FOR_TEST).copy()

download_results = []

for _, row in tqdm(download_df.iterrows(), total=len(download_df), desc="Descargando TLC Raw"):
    local_path = Path(row["local_path"])
    local_path.parent.mkdir(parents=True, exist_ok=True)
    part_path = local_path.with_suffix(local_path.suffix + ".part")

    result = {
        "pipeline_id": RUN_ID,
        "pipeline_name": PIPELINE_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "started_at": RUN_TS,
        "year": int(row["year"]),
        "month": int(row["month"]),
        "trip_type": row["trip_type"],
        "file_name": row["file_name"],
        "url": row["url"],
        "local_path": str(local_path),
        "status": None,
        "size_bytes": 0,
        "size_gb": 0.0,
        "sha256": None,
        "error": None,
    }

    try:
        if local_path.exists() and local_path.stat().st_size > 0 and not OVERWRITE_RAW:
            result["status"] = "skipped_existing"
        else:
            if part_path.exists():
                part_path.unlink()

            if has_curl():
                download_with_curl(row["url"], part_path)
            else:
                download_with_urllib(row["url"], part_path)

            if not part_path.exists() or part_path.stat().st_size == 0:
                raise RuntimeError("Descarga vacía.")

            part_path.replace(local_path)
            result["status"] = "downloaded"

        result["size_bytes"] = local_path.stat().st_size
        result["size_gb"] = round(result["size_bytes"] / (1024 ** 3), 4)
        result["sha256"] = sha256_file(local_path)

    except Exception as error:
        result["status"] = "failed"
        result["error"] = str(error)

        if part_path.exists():
            part_path.unlink()

    result["finished_at"] = datetime.now(ZoneInfo(LOCAL_TZ)).isoformat()
    append_manifest(result)
    download_results.append(result)

download_results_df = pd.DataFrame(download_results)

download_summary = (
    download_results_df
    .groupby("status", as_index=False)
    .agg(files=("file_name", "count"), gb=("size_gb", "sum"))
)

dataframe_table(download_summary, "Resultado de descarga", max_rows=20)

failed_downloads = download_results_df[download_results_df["status"] == "failed"]

if not failed_downloads.empty:
    dataframe_table(
        failed_downloads[["year", "month", "trip_type", "file_name", "error"]],
        "Errores de descarga",
        max_rows=30,
    )

    raise RuntimeError(f"Fallaron {len(failed_downloads)} descarga(s).")

panel("Manifest actualizado", MANIFEST_JSONL, "green")

Descargando TLC Raw: 100%|██████████| 4/4 [00:00<00:00,  6.95it/s]


              Resultado de descarga               
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ status           ┃ files ┃ gb                  ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ skipped_existing │ 4     │ 0.49750000000000005 │
└──────────────────┴───────┴─────────────────────┘

╭───────────────────────────────────────────── Manifest actualizado ──────────────────────────────────────────────╮
│ ..\..\data\logs\tlc_ingestion_manifest.jsonl                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [18]:
# ============================================================
# 6B. DESCARGA RAW - TAXI ZONE LOOKUP
# ============================================================

lookup_download_df = pd.DataFrame()

if INCLUDE_LOOKUP:
    lookup_result = {
        "pipeline_id": RUN_ID,
        "pipeline_name": PIPELINE_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "started_at": RUN_TS,
        "source_kind": "taxi_zone_lookup",
        "year": None,
        "month": None,
        "trip_type": "lookup",
        "file_name": "taxi_zone_lookup.csv",
        "url": LOOKUP_URL,
        "local_path": str(LOOKUP_RAW_PATH),
        "status": None,
        "size_bytes": 0,
        "size_gb": 0.0,
        "sha256": None,
        "error": None,
    }

    try:
        part_path = LOOKUP_RAW_PATH.with_suffix(LOOKUP_RAW_PATH.suffix + ".part")

        if LOOKUP_RAW_PATH.exists() and LOOKUP_RAW_PATH.stat().st_size > 0 and not OVERWRITE_RAW:
            lookup_result["status"] = "skipped_existing"

        else:
            if part_path.exists():
                part_path.unlink()

            if has_curl():
                download_with_curl(LOOKUP_URL, part_path)
            else:
                download_with_urllib(LOOKUP_URL, part_path)

            if not part_path.exists() or part_path.stat().st_size == 0:
                raise RuntimeError("Descarga vacía del Taxi Zone Lookup.")

            part_path.replace(LOOKUP_RAW_PATH)
            lookup_result["status"] = "downloaded"

        lookup_result["size_bytes"] = int(LOOKUP_RAW_PATH.stat().st_size)
        lookup_result["size_gb"] = round(lookup_result["size_bytes"] / (1024 ** 3), 6)
        lookup_result["sha256"] = sha256_file(LOOKUP_RAW_PATH)

    except Exception as error:
        lookup_result["status"] = "failed"
        lookup_result["error"] = str(error)

        part_path = LOOKUP_RAW_PATH.with_suffix(LOOKUP_RAW_PATH.suffix + ".part")
        if part_path.exists():
            part_path.unlink()

    lookup_result["finished_at"] = datetime.now(ZoneInfo(LOCAL_TZ)).isoformat()

    append_manifest(lookup_result)

    lookup_download_df = pd.DataFrame([lookup_result])

    dataframe_table(
        lookup_download_df[[
            "source_kind",
            "file_name",
            "status",
            "size_bytes",
            "sha256",
            "error",
        ]],
        "Descarga Taxi Zone Lookup",
        max_rows=5,
    )

    if lookup_result["status"] == "failed":
        raise RuntimeError("Falló la descarga del Taxi Zone Lookup.")

                                             Descarga Taxi Zone Lookup                                             
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ source_kind      ┃ file_name            ┃ status     ┃ size_bytes ┃ sha256                              ┃ error ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ taxi_zone_lookup │ taxi_zone_lookup.csv │ downloaded │ 12331      │ 1a99e105092230f8620f301edcca7f80d30 │       │
│                  │                      │            │            │ 80642ff404d28ed957d3fa222c8ed       │       │
└──────────────────┴──────────────────────┴────────────┴────────────┴─────────────────────────────────────┴───────┘

In [19]:
# ============================================================
# 7. METADATA VALIDATOR
# ============================================================

import pyarrow.parquet as pq

def parquet_signature_ok(file_path):
    path = Path(file_path)

    if not path.exists() or path.stat().st_size < 8:
        return False

    with open(path, "rb") as file:
        start = file.read(4)
        file.seek(-4, os.SEEK_END)
        end = file.read(4)

    return start == b"PAR1" and end == b"PAR1"

def spark_read_path_candidates(path):
    path = Path(path).resolve()
    candidates = []

    candidates.append(str(path))
    candidates.append(path.as_uri())

    if platform.system().lower() == "windows":
        candidates.append("file:///" + path.as_posix().replace(":", ":", 1))

    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)

    return unique

def read_schema_with_spark(path):
    last_error = None

    for candidate in spark_read_path_candidates(path):
        try:
            df_spark = spark.read.parquet(candidate)
            return {
                "spark_status": "valid",
                "spark_schema_json": df_spark.schema.json(),
                "spark_read_path": candidate,
                "spark_error": None,
            }
        except Exception as error:
            last_error = str(error)

    return {
        "spark_status": "invalid",
        "spark_schema_json": None,
        "spark_read_path": None,
        "spark_error": last_error,
    }

available_files_df = catalog_df.copy()
available_files_df["exists_local"] = available_files_df["local_path"].apply(lambda value: Path(value).exists())
available_files_df = available_files_df[available_files_df["exists_local"]].copy()

metadata_records = []

for _, row in tqdm(available_files_df.iterrows(), total=len(available_files_df), desc="Validando metadata"):
    file_path = Path(row["local_path"])

    result = {
        "pipeline_id": RUN_ID,
        "year": int(row["year"]),
        "month": int(row["month"]),
        "trip_type": row["trip_type"],
        "file_name": row["file_name"],
        "local_path": str(file_path),
        "validation_status": "invalid",
        "size_bytes": 0,
        "size_gb": 0.0,
        "parquet_rows": None,
        "sha256": None,
        "spark_schema_json": None,
        "spark_read_path": None,
        "validation_error": None,
    }

    try:
        if not file_path.exists():
            raise RuntimeError("El archivo no existe localmente.")

        size_bytes = file_path.stat().st_size

        if size_bytes == 0:
            raise RuntimeError("El archivo está vacío.")

        if file_path.suffix.lower() != ".parquet":
            raise RuntimeError("El archivo no tiene extensión .parquet.")

        if not parquet_signature_ok(file_path):
            raise RuntimeError("La firma Parquet no es válida.")

        pq_file = pq.ParquetFile(file_path)

        result["size_bytes"] = int(size_bytes)
        result["size_gb"] = round(size_bytes / (1024 ** 3), 4)
        result["parquet_rows"] = int(pq_file.metadata.num_rows)
        result["sha256"] = sha256_file(file_path)

        spark_validation = read_schema_with_spark(file_path)

        result["spark_schema_json"] = spark_validation["spark_schema_json"]
        result["spark_read_path"] = spark_validation["spark_read_path"]

        if spark_validation["spark_status"] != "valid":
            raise RuntimeError("PySpark no pudo leer el archivo: " + str(spark_validation["spark_error"]))

        result["validation_status"] = "valid"

    except Exception as error:
        result["validation_error"] = str(error)

    metadata_records.append(result)

metadata_df = pd.DataFrame(metadata_records)

dataframe_table(
    metadata_df[[
        "year",
        "month",
        "trip_type",
        "file_name",
        "validation_status",
        "size_gb",
        "parquet_rows",
        "validation_error",
    ]],
    "Metadata validada",
    max_rows=60,
)

invalid_df = metadata_df[metadata_df["validation_status"] != "valid"]

if not invalid_df.empty:
    dataframe_table(
        invalid_df[["year", "month", "trip_type", "file_name", "validation_error"]],
        "Errores de validación",
        max_rows=60,
    )

    raise RuntimeError(f"Hay {len(invalid_df)} archivo(s) inválidos.")

valid_metadata_df = metadata_df[metadata_df["validation_status"] == "valid"].copy()
panel("Metadata validator", f"Archivos válidos: {len(valid_metadata_df)}", "green")

Validando metadata: 100%|██████████| 4/4 [00:00<00:00,  4.57it/s]


                                                 Metadata validada                                                 
┏━━━━━━┳━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ year ┃ month ┃ trip_type ┃ file_name            ┃ validation_status ┃ size_gb ┃ parquet_rows ┃ validation_error ┃
┡━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 2023 │ 1     │ yellow    │ yellow_tripdata_2023 │ valid             │ 0.0444  │ 3066766      │                  │
│      │       │           │ -01.parquet          │                   │         │              │                  │
│ 2023 │ 1     │ green     │ green_tripdata_2023- │ valid             │ 0.0013  │ 68211        │                  │
│      │       │           │ 01.parquet           │                   │         │              │                  │
│ 2023 │ 1     │ fhv       │ fhv_tripdata_2023-01 │ valid             │ 0.0105  │ 1114320      │                  │
│      │       │           │ .parquet             │                   │         │              │                  │
│ 2023 │ 1     │ fhvhv     │ fhvhv_tripdata_2023- │ valid             │ 0.4413  │ 18479031     │                  │
│      │       │           │ 01.parquet           │                   │         │              │                  │
└──────┴───────┴───────────┴──────────────────────┴───────────────────┴─────────┴──────────────┴──────────────────┘

╭────────────────────────────────────────────── Metadata validator ───────────────────────────────────────────────╮
│ Archivos válidos: 4                                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [20]:
# ============================================================
# 7B. METADATA VALIDATOR - TAXI ZONE LOOKUP
# ============================================================

import csv

lookup_metadata_df = pd.DataFrame()

if INCLUDE_LOOKUP:
    lookup_validation = {
        "pipeline_id": RUN_ID,
        "source_kind": "taxi_zone_lookup",
        "file_name": "taxi_zone_lookup.csv",
        "local_path": str(LOOKUP_RAW_PATH),
        "validation_status": "invalid",
        "size_bytes": 0,
        "size_gb": 0.0,
        "records": None,
        "sha256": None,
        "validation_error": None,
    }

    try:
        if not LOOKUP_RAW_PATH.exists():
            raise RuntimeError("El Taxi Zone Lookup no existe localmente.")

        size_bytes = LOOKUP_RAW_PATH.stat().st_size

        if size_bytes == 0:
            raise RuntimeError("El Taxi Zone Lookup está vacío.")

        if LOOKUP_RAW_PATH.suffix.lower() != ".csv":
            raise RuntimeError("El Taxi Zone Lookup no tiene extensión .csv.")

        with open(LOOKUP_RAW_PATH, "r", encoding="utf-8-sig", newline="") as file:
            sample = file.read(2048)
            file.seek(0)
            csv.Sniffer().sniff(sample)

        lookup_df = pd.read_csv(LOOKUP_RAW_PATH)

        missing_columns = [
            column
            for column in LOOKUP_EXPECTED_COLUMNS
            if column not in lookup_df.columns
        ]

        if missing_columns:
            raise RuntimeError(f"Faltan columnas esperadas: {missing_columns}")

        if lookup_df.empty:
            raise RuntimeError("El Taxi Zone Lookup no contiene registros.")

        if lookup_df["LocationID"].duplicated().any():
            raise RuntimeError("LocationID contiene duplicados.")

        lookup_validation["validation_status"] = "valid"
        lookup_validation["size_bytes"] = int(size_bytes)
        lookup_validation["size_gb"] = round(size_bytes / (1024 ** 3), 6)
        lookup_validation["records"] = int(len(lookup_df))
        lookup_validation["sha256"] = sha256_file(LOOKUP_RAW_PATH)

    except Exception as error:
        lookup_validation["validation_error"] = str(error)

    lookup_metadata_df = pd.DataFrame([lookup_validation])

    dataframe_table(
        lookup_metadata_df[[
            "source_kind",
            "file_name",
            "validation_status",
            "records",
            "validation_error",
        ]],
        "Metadata Taxi Zone Lookup",
        max_rows=5,
    )

    if lookup_validation["validation_status"] != "valid":
        raise RuntimeError("Taxi Zone Lookup inválido.")

                                 Metadata Taxi Zone Lookup                                  
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ source_kind      ┃ file_name            ┃ validation_status ┃ records ┃ validation_error ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ taxi_zone_lookup │ taxi_zone_lookup.csv │ valid             │ 265     │                  │
└──────────────────┴──────────────────────┴───────────────────┴─────────┴──────────────────┘

In [21]:
# ============================================================
# 8. CARGA BRONZE
# ============================================================

import pyarrow as pa
import pyarrow.parquet as pq

bronze_audit_records = []

for _, row in valid_metadata_df.iterrows():
    input_path = Path(row["local_path"]).resolve()

    output_dir = (
        BRONZE_FILES_DIR
        / f"tipo_dataset={row['trip_type']}"
        / f"anio={int(row['year'])}"
        / f"mes={str(int(row['month'])).zfill(2)}"
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / row["file_name"]

    if output_file.exists():
        output_file.unlink()

    shutil.copy2(input_path, output_file)

    bronze_audit_records.append({
        "pipeline_id": RUN_ID,
        "pipeline_name": PIPELINE_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "fecha_ingesta": RUN_DATE,
        "hora_ingesta": RUN_TIME,
        "anio": int(row["year"]),
        "mes": int(row["month"]),
        "tipo_dataset": row["trip_type"],
        "nombre_archivo": row["file_name"],
        "raw_path": str(input_path),
        "bronze_path": str(output_file),
        "size_bytes": int(row["size_bytes"]),
        "size_gb": float(row["size_gb"]),
        "parquet_rows": int(row["parquet_rows"]),
        "hash_archivo": row["sha256"],
        "spark_schema_json": row["spark_schema_json"],
        "spark_read_path": row["spark_read_path"],
        "bronze_status": "loaded",
    })

bronze_audit_df = pd.DataFrame(bronze_audit_records)

bronze_audit_jsonl = BRONZE_METADATA_DIR / "bronze_audit.jsonl"
bronze_audit_parquet = BRONZE_METADATA_DIR / "bronze_audit.parquet"

bronze_audit_df.to_json(
    bronze_audit_jsonl,
    orient="records",
    lines=True,
    force_ascii=False,
)

pq.write_table(
    pa.Table.from_pandas(bronze_audit_df),
    bronze_audit_parquet,
)

dataframe_table(
    bronze_audit_df[[
        "anio",
        "mes",
        "tipo_dataset",
        "nombre_archivo",
        "parquet_rows",
        "bronze_status",
    ]],
    "Carga Bronze",
    max_rows=60,
)

panel("Bronze generado", BRONZE_DIR, "green")

                                         Carga Bronze                                         
┏━━━━━━┳━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ anio ┃ mes ┃ tipo_dataset ┃ nombre_archivo                  ┃ parquet_rows ┃ bronze_status ┃
┡━━━━━━╇━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ 2023 │ 1   │ yellow       │ yellow_tripdata_2023-01.parquet │ 3066766      │ loaded        │
│ 2023 │ 1   │ green        │ green_tripdata_2023-01.parquet  │ 68211        │ loaded        │
│ 2023 │ 1   │ fhv          │ fhv_tripdata_2023-01.parquet    │ 1114320      │ loaded        │
│ 2023 │ 1   │ fhvhv        │ fhvhv_tripdata_2023-01.parquet  │ 18479031     │ loaded        │
└──────┴─────┴──────────────┴─────────────────────────────────┴──────────────┴───────────────┘

╭──────────────────────────────────────────────── Bronze generado ────────────────────────────────────────────────╮
│ ..\..\data\bronze                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [22]:
# ============================================================
# 8B. CARGA BRONZE - TAXI ZONE LOOKUP
# ============================================================

lookup_bronze_audit_df = pd.DataFrame()

if INCLUDE_LOOKUP:
    if lookup_metadata_df.empty:
        raise RuntimeError("No existe metadata validada del Taxi Zone Lookup.")

    lookup_status = lookup_metadata_df.iloc[0]["validation_status"]

    if lookup_status != "valid":
        raise RuntimeError("No se puede cargar a Bronze un lookup inválido.")

    if LOOKUP_BRONZE_PATH.exists():
        LOOKUP_BRONZE_PATH.unlink()

    LOOKUP_BRONZE_PATH.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(LOOKUP_RAW_PATH, LOOKUP_BRONZE_PATH)

    lookup_bronze_audit_df = pd.DataFrame([{
        "pipeline_id": RUN_ID,
        "pipeline_name": PIPELINE_NAME,
        "pipeline_version": PIPELINE_VERSION,
        "fecha_ingesta": RUN_DATE,
        "hora_ingesta": RUN_TIME,
        "source_kind": "taxi_zone_lookup",
        "tipo_dataset": "lookup",
        "nombre_archivo": "taxi_zone_lookup.csv",
        "raw_path": str(LOOKUP_RAW_PATH.resolve()),
        "bronze_path": str(LOOKUP_BRONZE_PATH.resolve()),
        "size_bytes": int(lookup_metadata_df.iloc[0]["size_bytes"]),
        "size_gb": float(lookup_metadata_df.iloc[0]["size_gb"]),
        "records": int(lookup_metadata_df.iloc[0]["records"]),
        "hash_archivo": lookup_metadata_df.iloc[0]["sha256"],
        "bronze_status": "loaded",
    }])

    lookup_bronze_audit_jsonl = BRONZE_METADATA_DIR / "bronze_lookup_audit.jsonl"
    lookup_bronze_audit_parquet = BRONZE_METADATA_DIR / "bronze_lookup_audit.parquet"

    lookup_bronze_audit_df.to_json(
        lookup_bronze_audit_jsonl,
        orient="records",
        lines=True,
        force_ascii=False,
    )

    pq.write_table(
        pa.Table.from_pandas(lookup_bronze_audit_df),
        lookup_bronze_audit_parquet,
    )

    dataframe_table(
        lookup_bronze_audit_df[[
            "source_kind",
            "tipo_dataset",
            "nombre_archivo",
            "records",
            "bronze_status",
        ]],
        "Carga Bronze Taxi Zone Lookup",
        max_rows=5,
    )

    panel("Taxi Zone Lookup en Bronze", LOOKUP_BRONZE_PATH, "green")

                           Carga Bronze Taxi Zone Lookup                            
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ source_kind      ┃ tipo_dataset ┃ nombre_archivo       ┃ records ┃ bronze_status ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ taxi_zone_lookup │ lookup       │ taxi_zone_lookup.csv │ 265     │ loaded        │
└──────────────────┴──────────────┴──────────────────────┴─────────┴───────────────┘

╭────────────────────────────────────────── Taxi Zone Lookup en Bronze ───────────────────────────────────────────╮
│ ..\..\data\bronze\files\lookup\taxi_zone_lookup.csv                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [23]:
# ============================================================
# 9. RESUMEN FINAL
# ============================================================

final_summary = pd.DataFrame([
    ["pipeline_id", RUN_ID],
    ["raw_dir", RAW_DIR],
    ["bronze_files_dir", BRONZE_FILES_DIR],
    ["bronze_metadata_dir", BRONZE_METADATA_DIR],
    ["manifest_jsonl", MANIFEST_JSONL],
    ["archivos_catalogados", len(catalog_df)],
    ["archivos_descargados_o_existentes", len(download_results_df)],
    ["archivos_validados", len(valid_metadata_df)],
    ["archivos_bronze", len(bronze_audit_df)],
    ["bronze_audit_jsonl", bronze_audit_jsonl],
    ["bronze_audit_parquet", bronze_audit_parquet],
], columns=["métrica", "valor"])

dataframe_table(final_summary, "Resumen final", max_rows=30)
panel("Ejecución finalizada", "Nivel 2 y Nivel 3 completados.", "green")

                                     Resumen final                                      
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ métrica                           ┃ valor                                            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ pipeline_id                       │ tlc_ingestion_20260702_222700                    │
│ raw_dir                           │ ..\..\data\raw                                   │
│ bronze_files_dir                  │ ..\..\data\bronze\files                          │
│ bronze_metadata_dir               │ ..\..\data\bronze\_metadata                      │
│ manifest_jsonl                    │ ..\..\data\logs\tlc_ingestion_manifest.jsonl     │
│ archivos_catalogados              │ 48                                               │
│ archivos_descargados_o_existentes │ 4                                                │
│ archivos_validados                │ 4                                                │
│ archivos_bronze                   │ 4                                                │
│ bronze_audit_jsonl                │ ..\..\data\bronze\_metadata\bronze_audit.jsonl   │
│ bronze_audit_parquet              │ ..\..\data\bronze\_metadata\bronze_audit.parquet │
└───────────────────────────────────┴──────────────────────────────────────────────────┘

╭───────────────────────────────────────────── Ejecución finalizada ──────────────────────────────────────────────╮
│ Nivel 2 y Nivel 3 completados.                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯